# 📖 Goodreads Genre Classification — End-to-End MLOps Pipeline
### IIT Jodhpur PGD-AI MLOps Assignment 2

This notebook implements the complete MLOps workflow for fine-tuning a pre-trained **DistilBERT** model on the **UCSD Goodreads Reviews dataset** for 8-class genre classification.

#### Workflow Steps:
1. **Setup & Secrets:** Load API credentials (`WANDB_API_KEY`, `HF_TOKEN`) natively via Kaggle secrets.
2. **Data Pipeline:** Fetch, stream-sample, and split the balanced Goodreads dataset.
3. **Fine-Tuning:** Configure and execute the training loop using Hugging Face `Trainer` on Kaggle T4 GPUs, reporting live metrics to **Weights & Biases**.
4. **Evaluation:** Evaluate on the held-out test set, generate per-class classification reports, and upload reports to W&B as versioned Artifacts.
5. **HF Hub Model Push:** Programmatically upload final model weights and tokenizers to a public **Hugging Face Hub** repository.

## 🛠️ Step 1: Install Dependencies & Configure Kaggle Secrets

In [ ]:
# Install required libraries silently
!pip install -q transformers datasets wandb scikit-learn huggingface_hub

In [ ]:
import os
import sys

# Attempt to load Kaggle Secrets for W&B and HF authentication
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
    print("✅ Kaggle Secrets injected successfully!")
except Exception as e:
    print("⚠️ Kaggle secrets client not found or credentials missing.")
    print("Please ensure you add WANDB_API_KEY and HF_TOKEN under Add-ons -> Secrets.")
    # Fall back to checking environment variables (for local runs)
    if os.environ.get("WANDB_API_KEY") and os.environ.get("HF_TOKEN"):
        print("✅ Standard environment variables detected.")
    else:
        print("❌ Credentials missing. Make sure to define them before training!")

## 📦 Step 2: Shared Utilities (`utils.py`)
This block sets up the label mapping mappings and the custom PyTorch dataset helper.

In [ ]:
"""
utils.py
--------
Shared helpers used across data.py, train.py, and eval.py.

Contents
--------
- GENRES          : ordered list of genre class names
- label2id        : dict mapping genre name -> integer label
- id2label        : dict mapping integer label -> genre name
- NUM_LABELS      : total number of classes
- GoodreadsDataset: PyTorch Dataset wrapping tokenised reviews
- compute_metrics : evaluation function passed to HuggingFace Trainer
"""

from __future__ import annotations

import torch
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset
from transformers import PreTrainedTokenizerBase

# ---------------------------------------------------------------------------
# Label Maps
# ---------------------------------------------------------------------------
GENRES: list[str] = [
    "children",
    "comics_graphic",
    "fantasy_paranormal",
    "history_biography",
    "mystery_thriller_crime",
    "poetry",
    "romance",
    "young_adult",
]

label2id: dict[str, int] = {genre: idx for idx, genre in enumerate(GENRES)}
id2label: dict[int, str] = {idx: genre for genre, idx in label2id.items()}
NUM_LABELS: int = len(GENRES)


# ---------------------------------------------------------------------------
# Dataset Class
# ---------------------------------------------------------------------------
class GoodreadsDataset(Dataset):
    """
    PyTorch Dataset that tokenises Goodreads review texts on construction.

    Parameters
    ----------
    texts     : list[str]               Raw review strings.
    labels    : list[int]               Integer genre labels.
    tokenizer : PreTrainedTokenizerBase HuggingFace tokenizer instance.
    max_length: int                     Maximum token sequence length.
    """

    def __init__(
        self,
        texts: list[str],
        labels: list[int],
        tokenizer: PreTrainedTokenizerBase,
        max_length: int = 256,
    ) -> None:
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict:
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx],
        }


# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------
def compute_metrics(pred) -> dict[str, float]:
    """
    Compute accuracy and weighted F1 score from Trainer prediction output.

    Called automatically by the HuggingFace Trainer after each eval epoch.

    Parameters
    ----------
    pred : transformers.EvalPrediction
        Object with .label_ids (true labels) and .predictions (logits).

    Returns
    -------
    dict with keys 'accuracy' and 'f1'.
    """
    labels = pred.label_ids
    preds  = pred.predictions.argmax(axis=-1)
    return {
        "accuracy": round(accuracy_score(labels, preds), 4),
        "f1":       round(f1_score(labels, preds, average="weighted", zero_division=0), 4),
    }


## 📥 Step 3: Data Loading & Stream Sampling (`data.py`)
This block streams datasets from Mcauley Lab's official Goodreads repositories, samples exactly the requested number of records per genre, splits the balanced dataset, and tokenizes it.

In [ ]:
"""
data.py
-------
Data loading, balanced sampling, train/test split, and dataset encoding.

Usage
-----
    python data.py

Outputs (written to --output_dir):
    train_texts.json
    train_labels.json
    test_texts.json
    test_labels.json
"""

from __future__ import annotations

import argparse
import gzip
import io
import json
import os
import random
import sys
import warnings

import pandas as pd
import requests
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast

from utils import GENRES, GoodreadsDataset, label2id

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Constants (overridable via CLI)
# ---------------------------------------------------------------------------
DATASET_URL = (
    "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/"
    "goodreads_reviews_dedup.json.gz"
)
DEFAULT_SAMPLES_PER_GENRE = 500
DEFAULT_MAX_LENGTH        = 256
DEFAULT_TEST_SIZE         = 0.2
DEFAULT_SEED              = 42
DEFAULT_MODEL_NAME        = "distilbert-base-cased"
DEFAULT_OUTPUT_DIR        = "./data_cache"


# ---------------------------------------------------------------------------
# Download
# ---------------------------------------------------------------------------
def download_dataset(url: str, max_records_per_genre: int = 2000) -> pd.DataFrame:
    """
    Download the UCSD Goodreads reviews dataset and return as a DataFrame.

    If the url contains 'dedup', we automatically fall back to downloading
    and streaming by-genre files from the official UCSD Book Graph repository,
    since the dedup file does not contain a 'genre' column.

    Parameters
    ----------
    url                   : Direct URL or fallback indicator.
    max_records_per_genre : Number of records to stream per genre to guarantee sampling pool.

    Returns
    -------
    pd.DataFrame with 'genre' column injected.
    """
    if "dedup" in url:
        print("[INFO] Dedup dataset URL detected. Since the dedup dataset does not contain 'genre' columns,")
        print("       we are automatically streaming the by-genre datasets from Mcauley Lab.")
        
        all_dfs = []
        for genre in GENRES:
            genre_url = f"https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_{genre}.json.gz"
            print(f"Streaming genre: {genre} from:\n  {genre_url}")
            
            try:
                response = requests.get(genre_url, stream=True, timeout=30)
                response.raise_for_status()
                
                records = []
                with gzip.GzipFile(fileobj=response.raw) as gz_file:
                    for line in gz_file:
                        rec = json.loads(line)
                        if rec.get("review_text") and len(rec["review_text"].strip()) > 10:
                            rec["genre"] = genre
                            records.append(rec)
                            if len(records) >= max_records_per_genre:
                                break
                
                genre_df = pd.DataFrame(records)
                print(f"  Loaded {len(genre_df):,} records for {genre}.")
                all_dfs.append(genre_df)
            except Exception as e:
                print(f"[ERROR] Failed to stream genre {genre}: {e}")
                
        if not all_dfs:
            raise ValueError("Failed to download any genre files.")
        df = pd.concat(all_dfs, ignore_index=True)
        print(f"Successfully loaded a total of {len(df):,} records across all genres.")
        return df
    else:
        print(f"Downloading dataset from:\n  {url}")
        response = requests.get(url, stream=True, timeout=300)
        response.raise_for_status()

        records: list[dict] = []
        with gzip.GzipFile(fileobj=response.raw) as gz_file:
            for line in gz_file:
                records.append(json.loads(line))
                if len(records) >= 100_000:
                    break

        df = pd.DataFrame(records)
        print(f"Downloaded {len(df):,} records. Columns: {df.columns.tolist()}")
        return df


# ---------------------------------------------------------------------------
# Sampling
# ---------------------------------------------------------------------------
def sample_balanced(
    df: pd.DataFrame,
    samples_per_genre: int,
    text_col: str = "review_text",
    genre_col: str = "genre",
    seed: int = DEFAULT_SEED,
) -> pd.DataFrame:
    """
    Return a balanced subset with at most `samples_per_genre` rows per genre.

    Parameters
    ----------
    df                : Raw DataFrame from download_dataset.
    samples_per_genre : Maximum rows per genre class.
    text_col          : Column name containing review text.
    genre_col         : Column name containing genre label strings.
    seed              : Random seed for reproducibility.

    Returns
    -------
    Shuffled DataFrame with columns ['text', 'label'].
    """
    frames: list[pd.DataFrame] = []
    for genre in GENRES:
        subset = df[df[genre_col] == genre]
        n = min(samples_per_genre, len(subset))
        if n == 0:
            print(f"[WARNING] No samples found for genre: {genre}")
            continue
        sample = subset.sample(n=n, random_state=seed)[[text_col, genre_col]].copy()
        sample.rename(columns={text_col: "text"}, inplace=True)
        sample["label"] = label2id[genre]
        frames.append(sample)

    if not frames:
        raise ValueError(
            "No samples were found for any genre. "
            "Check that the genre_col and GENRES list match the dataset."
        )

    combined = (
        pd.concat(frames, ignore_index=True)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )
    combined = combined[["text", "label"]].dropna()
    combined["text"] = combined["text"].astype(str).str.strip()
    combined = combined[combined["text"].str.len() > 10].reset_index(drop=True)

    print(f"Balanced dataset: {len(combined):,} rows")
    print("Class distribution:")
    from utils import id2label  # local import to avoid circular at module level
    print(combined["label"].map(id2label).value_counts().to_string())
    return combined


# ---------------------------------------------------------------------------
# Split
# ---------------------------------------------------------------------------
def split_dataset(
    df: pd.DataFrame,
    test_size: float = DEFAULT_TEST_SIZE,
    seed: int = DEFAULT_SEED,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Stratified train/test split.

    Parameters
    ----------
    df        : Balanced DataFrame with 'text' and 'label' columns.
    test_size : Fraction of data held out for evaluation.
    seed      : Random seed.

    Returns
    -------
    (train_df, test_df)
    """
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df["label"],
    )
    print(f"Train: {len(train_df):,}  |  Test: {len(test_df):,}")
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


# ---------------------------------------------------------------------------
# Encode
# ---------------------------------------------------------------------------
def build_datasets(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    model_name: str = DEFAULT_MODEL_NAME,
    max_length: int = DEFAULT_MAX_LENGTH,
) -> tuple[GoodreadsDataset, GoodreadsDataset, DistilBertTokenizerFast]:
    """
    Tokenise DataFrames and return PyTorch Dataset objects.

    Parameters
    ----------
    train_df   : Training DataFrame with 'text' and 'label' columns.
    test_df    : Test DataFrame.
    model_name : HuggingFace model identifier.
    max_length : Maximum token sequence length.

    Returns
    -------
    (train_dataset, test_dataset, tokenizer)
    """
    print(f"Loading tokenizer: {model_name}")
    tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

    print("Tokenising train set...")
    train_dataset = GoodreadsDataset(
        train_df["text"].tolist(),
        train_df["label"].tolist(),
        tokenizer,
        max_length,
    )

    print("Tokenising test set...")
    test_dataset = GoodreadsDataset(
        test_df["text"].tolist(),
        test_df["label"].tolist(),
        tokenizer,
        max_length,
    )

    print(f"Train dataset: {len(train_dataset)} samples")
    print(f"Test dataset:  {len(test_dataset)} samples")
    return train_dataset, test_dataset, tokenizer


# ---------------------------------------------------------------------------
# Serialise raw splits for re-use across runs (optional)
# ---------------------------------------------------------------------------
def save_splits(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    output_dir: str,
) -> None:
    """Save text/label lists as JSON so train.py can reload without re-downloading."""
    os.makedirs(output_dir, exist_ok=True)
    for name, df in (("train", train_df), ("test", test_df)):
        with open(os.path.join(output_dir, f"{name}_texts.json"), "w") as f:
            json.dump(df["text"].tolist(), f)
        with open(os.path.join(output_dir, f"{name}_labels.json"), "w") as f:
            json.dump(df["label"].tolist(), f)
    print(f"Splits saved to: {output_dir}")


def load_splits(output_dir: str) -> tuple[list[str], list[int], list[str], list[int]]:
    """Load previously saved text/label lists."""
    def _load(fname):
        with open(os.path.join(output_dir, fname)) as f:
            return json.load(f)

    return (
        _load("train_texts.json"),
        _load("train_labels.json"),
        _load("test_texts.json"),
        _load("test_labels.json"),
    )


# ---------------------------------------------------------------------------
# CLI entry point
# ---------------------------------------------------------------------------
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Download, sample, split, and encode the Goodreads dataset."
    )
    parser.add_argument("--samples_per_genre", type=int, default=DEFAULT_SAMPLES_PER_GENRE)
    parser.add_argument("--max_length",        type=int, default=DEFAULT_MAX_LENGTH)
    parser.add_argument("--test_size",         type=float, default=DEFAULT_TEST_SIZE)
    parser.add_argument("--seed",              type=int, default=DEFAULT_SEED)
    parser.add_argument("--model_name",        type=str, default=DEFAULT_MODEL_NAME)
    parser.add_argument("--output_dir",        type=str, default=DEFAULT_OUTPUT_DIR)
    parser.add_argument("--dataset_url",       type=str, default=DATASET_URL)
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    random.seed(args.seed)

    max_rec = max(args.samples_per_genre * 3, 2000)
    df_raw  = download_dataset(args.dataset_url, max_records_per_genre=max_rec)
    df      = sample_balanced(df_raw, args.samples_per_genre, seed=args.seed)
    train_df, test_df = split_dataset(df, args.test_size, args.seed)
    save_splits(train_df, test_df, args.output_dir)

    # Quick tokenisation check
    build_datasets(train_df, test_df, args.model_name, args.max_length)
    print("data.py completed successfully.")


if __name__ == "__main__":
    main()


## 🚀 Step 4: Model Loaders & Training Arguments Setup (`train.py`)
This block defines model initialization, sets AdamW arguments, sets up the trainer, and trains on GPU.

In [ ]:
"""
train.py
--------
Model loading, Weights & Biases initialisation, Trainer configuration,
and the training loop.

Usage
-----
    python train.py [options]

Required environment variables
-------------------------------
    WANDB_API_KEY  — from https://wandb.ai/settings
    HF_TOKEN       — from https://huggingface.co/settings/tokens

Options
-------
    --data_dir        Directory written by data.py  (default: ./data_cache)
    --output_dir      Where checkpoints are saved   (default: ./results)
    --model_name      HuggingFace model id           (default: distilbert-base-cased)
    --epochs          Number of training epochs      (default: 3)
    --batch_size      Per-device train batch size    (default: 16)
    --eval_batch      Per-device eval batch size     (default: 32)
    --lr              Learning rate                  (default: 3e-5)
    --warmup_steps    LR scheduler warm-up steps     (default: 100)
    --weight_decay    AdamW weight decay             (default: 0.01)
    --max_length      Max tokenisation length        (default: 256)
    --wandb_project   W&B project name               (default: mlops-assignment2)
    --wandb_run       W&B run name                   (default: distilbert-run-1)
    --seed            Random seed                    (default: 42)
    --no_fp16         Disable mixed-precision even on GPU
"""

from __future__ import annotations

import argparse
import os
import random

import numpy as np
import torch
import wandb
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

from data import build_datasets, load_splits
from utils import NUM_LABELS, compute_metrics, id2label, label2id


# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
def load_model(
    model_name: str,
    device: torch.device,
) -> DistilBertForSequenceClassification:
    """
    Load DistilBERT (or any compatible model) with the correct classification head.

    Parameters
    ----------
    model_name : HuggingFace model identifier.
    device     : torch.device to move the model to.

    Returns
    -------
    Model ready for fine-tuning.
    """
    print(f"Loading model: {model_name}  |  num_labels={NUM_LABELS}")
    model = DistilBertForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id,
    ).to(device)

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters:     {total:,}")
    print(f"Trainable parameters: {trainable:,}")
    return model


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
def run_training(args: argparse.Namespace) -> Trainer:
    """
    Full training pipeline: seed -> data -> model -> W&B -> Trainer.train().

    Parameters
    ----------
    args : Parsed CLI arguments (see parse_args).

    Returns
    -------
    Trainer instance after training (holds the best checkpoint).
    """
    set_seed(args.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if torch.cuda.is_available():
        print(f"GPU:    {torch.cuda.get_device_name(0)}")

    # --- Data ---
    train_texts, train_labels, test_texts, test_labels = load_splits(args.data_dir)
    tokenizer = DistilBertTokenizerFast.from_pretrained(args.model_name)

    from utils import GoodreadsDataset
    train_dataset = GoodreadsDataset(train_texts, train_labels, tokenizer, args.max_length)
    test_dataset  = GoodreadsDataset(test_texts,  test_labels,  tokenizer, args.max_length)

    print(f"Train samples: {len(train_dataset):,}")
    print(f"Test samples:  {len(test_dataset):,}")

    # --- Model ---
    model = load_model(args.model_name, device)

    # --- W&B ---
    wandb.login(key=os.environ.get("WANDB_API_KEY", ""))
    run = wandb.init(
        project=args.wandb_project,
        name=args.wandb_run,
        config={
            "model":          args.model_name,
            "epochs":         args.epochs,
            "batch_size":     args.batch_size,
            "learning_rate":  args.lr,
            "max_length":     args.max_length,
            "warmup_steps":   args.warmup_steps,
            "weight_decay":   args.weight_decay,
            "dataset":        "UCSD Goodreads",
            "platform":       "local / Kaggle",
            "num_labels":     NUM_LABELS,
            "train_samples":  len(train_dataset),
            "test_samples":   len(test_dataset),
            "seed":           args.seed,
        },
        tags=["distilbert", "text-classification", "goodreads"],
    )
    print(f"W&B run: {run.url}")

    # --- Training Arguments ---
    use_fp16 = (not args.no_fp16) and torch.cuda.is_available()
    training_args = TrainingArguments(
        output_dir=args.output_dir,

        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.eval_batch,
        warmup_steps=args.warmup_steps,
        weight_decay=args.weight_decay,
        learning_rate=args.lr,

        logging_dir=os.path.join(args.output_dir, "logs"),
        logging_steps=50,

        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,

        report_to="wandb",          # single line enables full W&B logging
        run_name=args.wandb_run,

        seed=args.seed,
        fp16=use_fp16,
    )

    print(f"Training Arguments:")
    print(f"  Epochs:      {training_args.num_train_epochs}")
    print(f"  LR:          {training_args.learning_rate}")
    print(f"  report_to:   {training_args.report_to}")
    print(f"  fp16:        {training_args.fp16}")

    # --- Trainer ---
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    print("\nStarting training...")
    train_result = trainer.train()
    print("\nTraining completed.")
    print(f"  Runtime:          {train_result.metrics['train_runtime']:.1f} s")
    print(f"  Samples/sec:      {train_result.metrics['train_samples_per_second']:.1f}")
    print(f"  Final train loss: {train_result.metrics['train_loss']:.4f}")

    # Save the best model weights and the tokenizer to the output directory root
    print(f"Saving final model and tokenizer to: {args.output_dir}")
    trainer.save_model(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

    return trainer


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Fine-tune DistilBERT on Goodreads genre classification."
    )
    parser.add_argument("--data_dir",       type=str,   default="./data_cache")
    parser.add_argument("--output_dir",     type=str,   default="./results")
    parser.add_argument("--model_name",     type=str,   default="distilbert-base-cased")
    parser.add_argument("--epochs",         type=int,   default=3)
    parser.add_argument("--batch_size",     type=int,   default=16)
    parser.add_argument("--eval_batch",     type=int,   default=32)
    parser.add_argument("--lr",             type=float, default=3e-5)
    parser.add_argument("--warmup_steps",   type=int,   default=100)
    parser.add_argument("--weight_decay",   type=float, default=0.01)
    parser.add_argument("--max_length",     type=int,   default=256)
    parser.add_argument("--wandb_project",  type=str,   default="mlops-assignment2")
    parser.add_argument("--wandb_run",      type=str,   default="distilbert-run-1")
    parser.add_argument("--seed",           type=int,   default=42)
    parser.add_argument(
        "--no_fp16",
        action="store_true",
        help="Disable mixed-precision training even if a GPU is available.",
    )
    return parser.parse_args()


def main() -> None:
    args   = parse_args()
    trainer = run_training(args)
    # Keep trainer available for eval.py when used as a module
    return trainer


if __name__ == "__main__":
    main()
    wandb.finish()
    print("train.py completed successfully.")


## 📊 Step 5: Test Evaluation & W&B Artifacts Upload (`eval.py`)
This block runs the final validation on the held-out test set, displays the classification report, and registers the report as a W&B versioned Artifact.

In [ ]:
"""
eval.py
-------
Final evaluation on the test set, metric logging to W&B, and saving the
classification report as a versioned W&B Artifact.

Usage
-----
    # Run after train.py has saved a checkpoint to --model_dir
    python eval.py [options]

Required environment variables
-------------------------------
    WANDB_API_KEY  — from https://wandb.ai/settings

Options
-------
    --model_dir     Path to the saved model/tokenizer  (default: ./results)
    --data_dir      Directory written by data.py        (default: ./data_cache)
    --output_dir    Where eval_report.json is saved     (default: ./results)
    --max_length    Max tokenisation length             (default: 256)
    --eval_batch    Per-device eval batch size          (default: 32)
    --wandb_project W&B project name                    (default: mlops-assignment2)
    --wandb_run     W&B run name for the eval run       (default: distilbert-eval)
    --seed          Random seed                         (default: 42)
"""

from __future__ import annotations

import argparse
import json
import os

import torch
import wandb
from sklearn.metrics import classification_report
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    Trainer,
    TrainingArguments,
)

from data import load_splits
from utils import GoodreadsDataset, NUM_LABELS, compute_metrics, id2label, label2id


# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------
def evaluate(args: argparse.Namespace) -> dict:
    """
    Load a fine-tuned model, run evaluation on the test set, log metrics to
    W&B, and upload the classification report as a W&B Artifact.

    Parameters
    ----------
    args : Parsed CLI arguments (see parse_args).

    Returns
    -------
    dict containing eval_loss, eval_accuracy, and eval_f1.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # --- Load model and tokenizer ---
    print(f"Loading model from: {args.model_dir}")
    tokenizer = DistilBertTokenizerFast.from_pretrained(args.model_dir)
    model = DistilBertForSequenceClassification.from_pretrained(
        args.model_dir,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id,
    ).to(device)

    # --- Load test data ---
    _, _, test_texts, test_labels = load_splits(args.data_dir)
    test_dataset = GoodreadsDataset(test_texts, test_labels, tokenizer, args.max_length)
    print(f"Test samples: {len(test_dataset):,}")

    # --- Minimal TrainingArguments for inference (no training) ---
    eval_args = TrainingArguments(
        output_dir=args.output_dir,
        per_device_eval_batch_size=args.eval_batch,
        report_to="none",    # W&B logging done manually below
        seed=args.seed,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=eval_args,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
    )

    # --- Run evaluation ---
    print("Running evaluation on test set...")
    eval_results = trainer.evaluate(eval_dataset=test_dataset)

    final_accuracy = eval_results.get("eval_accuracy", 0.0)
    final_f1       = eval_results.get("eval_f1",       0.0)
    final_loss     = eval_results.get("eval_loss",     0.0)

    print("=" * 42)
    print("     FINAL EVALUATION METRICS")
    print("=" * 42)
    print(f"  Accuracy:  {final_accuracy:.4f}  ({final_accuracy * 100:.2f}%)")
    print(f"  F1 Score:  {final_f1:.4f}")
    print(f"  Eval Loss: {final_loss:.4f}")
    print("=" * 42)

    # --- W&B: log final metrics explicitly ---
    wandb.login(key=os.environ.get("WANDB_API_KEY", ""))
    wandb.init(
        project=args.wandb_project,
        name=args.wandb_run,
        config={
            "model_dir":   args.model_dir,
            "max_length":  args.max_length,
            "num_labels":  NUM_LABELS,
        },
    )

    wandb.log({
        "final/loss":     final_loss,
        "final/accuracy": final_accuracy,
        "final/f1":       final_f1,
    })

    wandb.run.summary["best_accuracy"] = final_accuracy
    wandb.run.summary["best_f1"]       = final_f1
    wandb.run.summary["best_loss"]     = final_loss

    print("Final metrics logged to W&B.")

    # --- Per-class classification report ---
    pred_output = trainer.predict(test_dataset)
    pred_labels = pred_output.predictions.argmax(axis=-1)
    true_labels = [test_dataset[i]["labels"].item() for i in range(len(test_dataset))]

    print("\nPer-class Classification Report:")
    print(
        classification_report(
            true_labels,
            pred_labels,
            target_names=[id2label[i] for i in range(NUM_LABELS)],
            zero_division=0,
        )
    )

    report_dict = classification_report(
        true_labels,
        pred_labels,
        target_names=[id2label[i] for i in range(NUM_LABELS)],
        output_dict=True,
        zero_division=0,
    )

    # --- Save report as JSON ---
    os.makedirs(args.output_dir, exist_ok=True)
    report_path = os.path.join(args.output_dir, "eval_report.json")
    with open(report_path, "w") as f:
        json.dump(report_dict, f, indent=2)
    print(f"Classification report saved to: {report_path}")

    # --- Upload as W&B Artifact ---
    artifact = wandb.Artifact(
        name="eval-report",
        type="evaluation",
        description="Per-class classification report from final test set evaluation.",
        metadata={
            "accuracy": final_accuracy,
            "f1":       final_f1,
            "loss":     final_loss,
        },
    )
    artifact.add_file(report_path)
    wandb.log_artifact(artifact)
    print("Artifact uploaded to W&B.")

    wandb.finish()
    return eval_results


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Evaluate a fine-tuned model and log results to W&B."
    )
    parser.add_argument("--model_dir",      type=str, default="./results")
    parser.add_argument("--data_dir",       type=str, default="./data_cache")
    parser.add_argument("--output_dir",     type=str, default="./results")
    parser.add_argument("--max_length",     type=int, default=256)
    parser.add_argument("--eval_batch",     type=int, default=32)
    parser.add_argument("--wandb_project",  type=str, default="mlops-assignment2")
    parser.add_argument("--wandb_run",      type=str, default="distilbert-eval")
    parser.add_argument("--seed",           type=int, default=42)
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    evaluate(args)
    print("eval.py completed successfully.")


if __name__ == "__main__":
    main()


## 🏃‍♂️ Step 6: End-to-End Orchestrator (Run the Pipeline)
Define our hyperparameter values, execute data fetching, split generation, fine-tuning, and metrics collection in one click.

In [ ]:
# ---------------------------------------------------------------------------
# End-to-End Execution Flow
# ---------------------------------------------------------------------------
import os
import torch
import wandb

SAMPLES_PER_GENRE = 500  # Target fine-tuning dataset size per genre class
EPOCHS            = 3
BATCH_SIZE        = 16
EVAL_BATCH        = 32
MAX_LENGTH        = 256
LEARNING_RATE     = 3e-5
SEED              = 42

DATA_CACHE_DIR    = "./data_cache"
RESULTS_DIR       = "./results"
MODEL_NAME        = "distilbert-base-cased"
WANDB_PROJECT     = "mlops-assignment2"
WANDB_RUN_NAME    = "distilbert-t4-gpu-run"

# Seed for reproducibility
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("🚀 Starting Goodreads Genre Classification Pipeline...")

# --- 1. Data Prep ---
max_rec = max(SAMPLES_PER_GENRE * 3, 2000)
df_raw  = download_dataset("goodreads_reviews_dedup.json.gz", max_records_per_genre=max_rec)
df      = sample_balanced(df_raw, SAMPLES_PER_GENRE, seed=SEED)
train_df, test_df = split_dataset(df, test_size=0.2, seed=SEED)
save_splits(train_df, test_df, DATA_CACHE_DIR)

# --- 2. Tokenisation & Encoding ---
train_dataset, test_dataset, tokenizer = build_datasets(
    train_df, test_df, model_name=MODEL_NAME, max_length=MAX_LENGTH
)

# --- 3. Model Fine-Tuning ---
class DummyArgs:
    def __init__(self):
        self.data_dir = DATA_CACHE_DIR
        self.output_dir = RESULTS_DIR
        self.model_name = MODEL_NAME
        self.epochs = EPOCHS
        self.batch_size = BATCH_SIZE
        self.eval_batch = EVAL_BATCH
        self.lr = LEARNING_RATE
        self.warmup_steps = 100
        self.weight_decay = 0.01
        self.max_length = MAX_LENGTH
        self.wandb_project = WANDB_PROJECT
        self.wandb_run = WANDB_RUN_NAME
        self.seed = SEED
        self.no_fp16 = False

train_args = DummyArgs()
trainer = run_training(train_args)
wandb.finish()
print("✅ Fine-Tuning Completed successfully!")

## 📐 Step 7: Final Test Evaluation
Call our evaluation helper to obtain clean classification scores and upload W&B evaluation Artifacts.

In [ ]:
class DummyEvalArgs:
    def __init__(self):
        self.model_dir = RESULTS_DIR
        self.data_dir = DATA_CACHE_DIR
        self.output_dir = RESULTS_DIR
        self.max_length = MAX_LENGTH
        self.eval_batch = EVAL_BATCH
        self.wandb_project = WANDB_PROJECT
        self.wandb_run = WANDB_RUN_NAME + "-eval"
        self.seed = SEED

eval_args = DummyEvalArgs()
evaluate(eval_args)
print("✅ Evaluation & Artifact registration complete!")

## 🤗 Step 8: Push Trained Weights and Tokenizer to Hugging Face Hub
Upload final fine-tuned model directly to Hugging Face Hub using your `HF_TOKEN`. Replace `your-username` with your own Hugging Face account name.

In [ ]:
from huggingface_hub import HfApi

# --- CONFIGURATION ---
HF_USERNAME = "your-username"  # REPLACE with your real HF username
REPO_NAME   = "distilbert-goodreads-genres"

repo_id = f"{HF_USERNAME}/{REPO_NAME}"
hf_token = os.environ.get("HF_TOKEN", "")

if not hf_token:
    print("❌ HF_TOKEN not found in environment. Model could not be uploaded.")
    print("Make sure HF_TOKEN is injected using Add-ons -> Secrets.")
else:
    print(f"Initiating Hugging Face upload to repository: {repo_id}")
    try:
        api = HfApi(token=hf_token)
        
        # Create the repository if it doesn't already exist
        api.create_repo(repo_id=repo_id, exist_ok=True, private=False)
        
        # Upload the entire results directory containing weights, tokenizer, and config
        print("Uploading trained model directory...")
        api.upload_folder(
            folder_path=RESULTS_DIR,
            repo_id=repo_id,
            commit_message="fine-tune: upload best model weights and tokenizer from Kaggle T4 training"
        )
        print(f"🚀 SUCCESS! Your model has been successfully published to:")
        print(f"   👉 https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"❌ Failed to upload model: {e}")